# Train an SO101-Nexus policy with PPO on MuJoCo Warp

PPO on the GPU-parallel **MuJoCo Warp** backend. Pick a task in step 3.

Reference results on one RTX 5090: `WarpPickLift-v1` reaches 0.993 success in 31.6 min; touch, look-at, and move solve in about a minute. Colab A100/L4 are slower; a free T4 works but is slow (lower `--num-envs`).

## 1. Check the GPU

Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi -L

## 2. Install

The example scripts live in `examples/`, not the wheel, so clone the repo and install the `warp` + `train` extras.

In [ ]:
!git clone --depth 1 https://github.com/johnsutor/so101-nexus.git
%cd so101-nexus
!pip install -q -e ".[warp,train]" "imageio[ffmpeg]"

## 3. Choose the environment

PickLift needs the most steps; the others solve fast.

In [ ]:
# Options: WarpPickLift-v1, WarpTouch-v1, WarpLookAt-v1, WarpMove-v1
ENV_ID = "WarpPickLift-v1"
STEPS = {"WarpPickLift-v1": 100_000_000}.get(ENV_ID, 5_000_000)
print(f"Training {ENV_ID} for {STEPS:,} env steps")

## 4. Launch TensorBoard

Run before training so the dashboard picks up the new run.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Train

1024 GPU-parallel worlds, fixed-horizon episodes, entropy annealed 0.01 -> 0, staggered resets, obs/reward normalization. The entropy and LR schedules run over `--total-timesteps`, so the step budget is part of the recipe. `best_agent.pt` is saved on every success-rate improvement. Reduce `--num-envs` on smaller GPUs.

MuJoCo eval rendering is off on Colab (`--eval-freq 0`); step 6 evaluates on the Warp backend instead.

In [ ]:
!python examples/ppo_warp.py --exp-name colab --env-id {ENV_ID} \
    --total-timesteps {STEPS} --eval-freq 0

## 6. Evaluate the trained policy

Deterministic eval (no exploration noise) across 512 Warp worlds. Reports `success_rate(ever)`, hold rate at the final step, and mean return.

In [ ]:
CKPT = f"runs/{ENV_ID}__colab__*/best_agent.pt"
!python examples/eval_warp.py --env-id {ENV_ID} --checkpoint "{CKPT}"

## Next steps

- PickLift lift discovery is seed-sensitive; if a run stalls at low return / 0% success, try another `--seed`.
- Tuning: `examples/README.md` and the [Training with PPO](https://so101-nexus.com/docs/guides/training-with-ppo) guide.